In [ ]:
import json
import os
from pathlib import Path

# Absolute path to experiment_results.json
json_path = Path("/Users/mattia.sabella/PhD - PoliMi/Frog-DQ/Repository/FrogDQ.Federated_Proximal_Gating_with_Data_Quality/experiment_results.json")

# Read and extract first-level keys
if not json_path.exists():
    print(f"File not found: {json_path}")
    top_level_keys = []
else:
    text = json_path.read_text(encoding="utf-8")
    if not text.strip():
        print(f"File is empty: {json_path}")
        top_level_keys = []
    else:
        try:
            data = json.loads(text)
            if isinstance(data, dict):
                top_level_keys = list(data.keys())
            else:
                # If the top-level element isn't a dict, there are no 'object' keys
                # Optionally, handle list-of-dicts by inspecting the first element
                if isinstance(data, list) and data and isinstance(data[0], dict):
                    print("Top-level is a list; showing keys of the first element.")
                    top_level_keys = list(data[0].keys())
                else:
                    top_level_keys = []
                    print("Top-level JSON is not an object or list-of-objects; no keys to show.")
        except json.JSONDecodeError as e:
            print(f"Invalid JSON: {e}")
            top_level_keys = []

print("Top-level keys:", top_level_keys)


Invalid JSON: Expecting value: line 10211942 column 11 (char 318767104)
Context around error (chars 318766904..318767104):
,
            0.9304974675178528,
            1.1865100860595703,
            1.39008629322052,
            1.208462119102478,
            1.405792474746704,
            1.4131993055343628,
          
                                                                                                                                                                                                        ^
Offending line 10211942, col 11:
          
          ^
Could not parse as JSON Lines either.
Top-level keys: []


In [ ]:
from pathlib import Path
import json
from collections import defaultdict
from typing import List, Dict
import numpy as np
import matplotlib.pyplot as plt


def plot_learning_curves(
    dataset: str,
    data_poisoning_method: str,
    features_percentage: float,
    poisoning_percentage: float,
    model: str,
    plot_y: str,
) -> None:
    """
    Plot aggregated validation learning curves across FrogDQ modes.

    Parameters
    ----------
    dataset : str
        Dataset name used in experiments.
    data_poisoning_method : str
        Poisoning method (matches key "poisoning_method").
    features_percentage : float
        Percentage of features poisoned.
    poisoning_percentage : float
        Percentage of instances poisoned.
    model : str
        Model architecture (matches key "model_type").
    plot_y : {"loss", "accuracy", "balanced_accuracy", "auc"}
        Which validation metric to plot.
    """
    # Resolve experiment results path (repo root)
    results_path = Path(
        "/Users/mattia.sabella/PhD - PoliMi/Frog-DQ/Repository/FrogDQ.Federated_Proximal_Gating_with_Data_Quality/experiment_results.json"
    )

    if not results_path.exists():
        print(f"File not found: {results_path}")
        return

    # Read JSON with robust fallbacks
    try:
        text = results_path.read_text(encoding="utf-8")
    except Exception as e:
        print(f"Could not read results file: {e}")
        return

    if not text.strip():
        print("Results file is empty.")
        return

    try:
        payload = json.loads(text)
    except json.JSONDecodeError as e:
        print(f"Invalid JSON: {e}")
        return

    # Support two shapes: {"experiments": [...]} or a raw list [...]
    if isinstance(payload, dict) and "experiments" in payload and isinstance(payload["experiments"], list):
        experiments = payload["experiments"]
    elif isinstance(payload, list):
        experiments = payload
    else:
        print("Unrecognized results structure. Expecting {\"experiments\": [...]} or a list.")
        return

    # Map requested metric to history key
    metric_map = {
        "loss": "val_loss",
        "accuracy": "val_acc",
        "balanced_accuracy": "val_bal_acc",
        "auc": "val_auc",
    }
    if plot_y not in metric_map:
        print(f"Unsupported plot_y '{plot_y}'. Choose one of {list(metric_map)}")
        return
    history_key = metric_map[plot_y]

    # Filter experiments by parameters
    def _match(exp: Dict) -> bool:
        return (
            exp.get("dataset") == dataset
            and exp.get("poisoning_method") == data_poisoning_method
            and float(exp.get("features_percentage")) == float(features_percentage)
            and float(exp.get("poisoning_percentage")) == float(poisoning_percentage)
            and exp.get("model_type") == model
        )

    filtered: List[Dict] = [exp for exp in experiments if _match(exp)]

    if not filtered:
        print("No experiments matched the provided filters.")
        return

    # Group histories by frogdq_mode
    by_mode: Dict[str, List[List[float]]] = defaultdict(list)
    for exp in filtered:
        mode = exp.get("frogdq_mode", "unknown")
        hist = (exp.get("history") or {})
        seq = hist.get(history_key)
        if isinstance(seq, list) and len(seq) > 0:
            # Ensure numbers (convert None/NaN to np.nan, then drop for averaging step)
            cleaned = [float(x) if x is not None else np.nan for x in seq]
            by_mode[mode].append(cleaned)

    if not by_mode:
        print(f"No '{history_key}' sequences found for the matched experiments.")
        return

    # Compute element-wise mean over available values per epoch index
    def elementwise_mean(sequences: List[List[float]]) -> List[float]:
        max_len = max(len(s) for s in sequences)
        out: List[float] = []
        for i in range(max_len):
            vals = [s[i] for s in sequences if i < len(s) and not np.isnan(s[i])]
            if len(vals) == 0:
                # No available value at this index across sequences; repeat last if possible, else NaN
                out.append(out[-1] if out else np.nan)
            else:
                out.append(float(np.mean(vals)))
        return out

    # Plot
    plt.figure(figsize=(9, 5))
    modes_plotted = 0
    for mode, seqs in sorted(by_mode.items()):
        mean_curve = elementwise_mean(seqs)
        plt.plot(mean_curve, label=f"{mode} (n={len(seqs)})")
        modes_plotted += 1

    if modes_plotted == 0:
        print("Nothing to plot after aggregation.")
        return

    plt.title(
        f"Validation {plot_y} — dataset={dataset}, poison={data_poisoning_method}, "
        f"feat%={features_percentage}, pois%={poisoning_percentage}, model={model}"
    )
    plt.xlabel("Epoch")
    plt.ylabel(plot_y.replace("_", " ").title())
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

